# Capítulo 6 — Teoria de Carteiras, Asset Pricing e Performance

Este é o capítulo integrador: combinamos retornos, covariâncias, otimização, fatores e métricas de desempenho. A sequência conceitual é guiada pelo Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander, sem reproduzir o texto da obra.

**Objetivos:**

- relacionar utilidade, aversão ao risco e equivalente certo;
- construir carteiras GMV, Markowitz, tangência e fronteira eficiente;
- distinguir CML, SML, beta, alpha e risco residual;
- comparar Sharpe, Sortino, Omega, Kappa, Information Ratio, Jensen alpha e Treynor;
- entender como autocorrelação torna a anualização ingênua do Sharpe enganosa.

Todas as carteiras usam dados simulados e permitem comparar solução analítica com otimização numérica.

## Como ler este capítulo

Este capítulo integra as ferramentas anteriores. Em cada bloco siga: **Objetivo econômico**, **Intuição**, **Formulação**, **solução manual**, **solução com biblioteca**, **exemplo de carteira**, **visualização**, **interpretação**, **Armadilhas** e **Exercício**. Não confunda uma métrica descritiva com uma hipótese de asset pricing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize

from quantfinance.performance import (
    certainty_equivalent,
    information_ratio,
    kappa_ratio,
    omega_ratio,
    sharpe_ratio,
    sortino_ratio,
    treynor_ratio,
)
from quantfinance.portfolio import (
    capm_expected_return,
    efficient_frontier,
    estimate_capm,
    global_minimum_variance,
    jensen_alpha,
    minimum_variance_target_return,
    portfolio_sharpe,
    portfolio_variance,
    tangency_portfolio,
)

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 6.1 — Utilidade, aversão ao risco e equivalente certo

Uma função de utilidade $U(W)$ traduz preferências sobre riqueza. A aversão absoluta ao risco é

$$ARA(W)=-\frac{U''(W)}{U'(W)},$$

e a aversão relativa é

$$RRA(W)=-W\frac{U''(W)}{U'(W)}.$$

A tolerância ao risco é aproximadamente o inverso da aversão absoluta. Para utilidade exponencial,

$$U(W)=-e^{-aW}, \qquad ARA=a.$$

Sob o critério média-variância, o equivalente certo aproximado é

$$CE\approx E[R]-\frac{a}{2}\operatorname{Var}(R).$$

**Interpretação:** maior aversão reduz a riqueza certa equivalente para a mesma média e variância; a escolha depende da unidade e horizonte dos retornos.

**Exercício:** compare a utilidade e o equivalente certo para três níveis de aversão ao risco.

In [ ]:
def exponential_utility(wealth, risk_aversion=3.0):
    return -np.exp(-risk_aversion * wealth)


def absolute_risk_aversion(risk_aversion=3.0):
    return risk_aversion


def relative_risk_aversion(wealth, risk_aversion=3.0):
    return risk_aversion * wealth

wealth = np.linspace(0, 2, 300)
plt.figure(figsize=(8, 4))
for risk_aversion in [1.0, 3.0, 6.0]:
    plt.plot(wealth, exponential_utility(wealth, risk_aversion), label=f"a={risk_aversion}")
plt.title("Utilidade exponencial e aversão ao risco")
plt.xlabel("Riqueza")
plt.legend()
plt.show()

returns_for_ce = np.array([0.02, 0.01, -0.01, 0.03, 0.015])
for risk_aversion in [1.0, 3.0, 6.0]:
    print(
        "a=", risk_aversion,
        "ARA=", absolute_risk_aversion(risk_aversion),
        "RRA em W=1=", relative_risk_aversion(1.0, risk_aversion),
        "CE=", certainty_equivalent(returns_for_ce, risk_aversion),
    )

In [ ]:
mu_return = returns_for_ce.mean()
variance_return = returns_for_ce.var(ddof=1)
for risk_aversion in [1.0, 3.0, 6.0]:
    ce_mean_variance = mu_return - 0.5 * risk_aversion * variance_return
    print("Critério média-variância, a=", risk_aversion, ":", ce_mean_variance)

## 6.2 — Diversificação, covariância e GMV

A diversificação depende da covariância, não apenas do número de ativos. Para pesos que somam 1:

$$\sigma_p^2=\mathbf{w}'\Sigma\mathbf{w}.$$

A carteira global de mínima variância satisfaz

$$\mathbf{w}_{GMV}=\frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}'\Sigma^{-1}\mathbf{1}}.$$

Essa solução é analítica sem restrições. Com short selling proibido, limites individuais ou retorno-alvo, resolvemos um problema de otimização.

**Exercício:** compare GMV irrestrita, long-only e com limite máximo de 60% por ativo.

In [ ]:
expected_returns = np.array([0.05, 0.07, 0.09, 0.04, 0.08])
volatilities = np.array([0.15, 0.20, 0.25, 0.10, 0.18])
correlation = np.array([
    [1.00, 0.30, 0.10, 0.20, 0.25],
    [0.30, 1.00, 0.40, 0.15, 0.35],
    [0.10, 0.40, 1.00, 0.05, 0.20],
    [0.20, 0.15, 0.05, 1.00, 0.10],
    [0.25, 0.35, 0.20, 0.10, 1.00],
])
covariance = np.diag(volatilities) @ correlation @ np.diag(volatilities)
correlations = np.linspace(-1, 1, 200)
sigma1, sigma2, weight = 0.20, 0.40, 0.25
vols = np.sqrt(
    weight**2 * sigma1**2
    + (1 - weight)**2 * sigma2**2
    + 2 * correlations * weight * (1 - weight) * sigma1 * sigma2
)

plt.figure(figsize=(8, 4))
plt.plot(correlations, vols)
plt.xlabel("Correlação")
plt.ylabel("Volatilidade da carteira")
plt.title("Efeito da correlação sobre a diversificação")
plt.show()

## 6.3 — Markowitz, restrições e fronteira eficiente

A fronteira de Markowitz contém carteiras de variância mínima para cada retorno-alvo. A parte acima do GMV é a fronteira eficiente.

Vamos comparar:

1. fórmula analítica do GMV;
2. `scipy.optimize` com restrição long-only;
3. limites individuais e retorno-alvo.

**Exercício:** compare o efeito de permitir short selling com o caso long-only.

In [ ]:
ones = np.ones(expected_returns.size)
inverse_solution = np.linalg.inv(covariance) @ ones
analytic_gmv = inverse_solution / (ones @ inverse_solution)
package_gmv = global_minimum_variance(covariance)
long_only_gmv = global_minimum_variance(covariance, long_only=True)
bounded_gmv = global_minimum_variance(covariance, bounds=(0.0, 0.60))

print("GMV analítica:", analytic_gmv)
print("GMV pelo pacote:", package_gmv)
print("Equivalência analítica/pacote:", np.allclose(analytic_gmv, package_gmv))
print("GMV long-only:", long_only_gmv)
print("GMV com limite de 60%:", bounded_gmv)
print("Volatilidade GMV:", np.sqrt(portfolio_variance(package_gmv, covariance)))

## 6.4 — Fronteira eficiente: solução analítica e otimização

Para cada $\mu^*$, resolvemos:

$$\min_w\;w'\Sigma w$$

sujeito a

$$\mathbf{1}'w=1, \qquad \mu'w=\mu^*.$$

A versão analítica é conveniente sem limites; `scipy.optimize` permite impor long-only e bounds individuais.

**Interpretação:** a fronteira troca retorno esperado por risco; uma carteira fora dela é dominada por outra com menor risco para o mesmo retorno.

In [ ]:
target_returns = np.linspace(expected_returns.min(), expected_returns.max(), 40)
frontier = efficient_frontier(expected_returns, covariance, target_returns)
frontier_long_only = efficient_frontier(
    expected_returns, covariance, target_returns, long_only=True
)

# Comparação pontual com scipy.optimize.
target = 0.07
weights_target = minimum_variance_target_return(expected_returns, covariance, target)
scipy_result = minimize(
    lambda weights: portfolio_variance(weights, covariance),
    np.repeat(0.2, expected_returns.size),
    method="SLSQP",
    constraints=[
        {"type": "eq", "fun": lambda weights: weights.sum() - 1.0},
        {"type": "eq", "fun": lambda weights: weights @ expected_returns - target},
    ],
)
print("Pesos target-return pelo pacote:", weights_target)
print("Pesos target-return scipy:", scipy_result.x)
print("Retorno da carteira alvo:", weights_target @ expected_returns)

plt.figure(figsize=(8, 5))
plt.plot(frontier.volatilities, frontier.target_returns, label="fronteira analítica/otimização")
plt.plot(frontier_long_only.volatilities, frontier_long_only.target_returns, label="long-only")
plt.scatter(
    np.sqrt(portfolio_variance(package_gmv, covariance)),
    package_gmv @ expected_returns,
    label="GMV",
)
plt.xlabel("Volatilidade")
plt.ylabel("Retorno esperado")
plt.legend()
plt.title("Fronteira eficiente e restrição long-only")
plt.show()

## 6.5 — Ativo livre de risco, tangência e Capital Market Line

A carteira tangente maximiza o Sharpe entre ativos arriscados. Com um ativo livre de risco, combinações entre ele e a carteira tangente formam a Capital Market Line:

$$E(R_p)=R_f+S_T\sigma_p.$$

A CML relaciona **retorno esperado e volatilidade**. Ela não usa beta no eixo horizontal.

**Exercício:** compare a carteira tangente irrestrita com a long-only.

In [ ]:
risk_free = 0.03
tangency = tangency_portfolio(expected_returns, covariance, risk_free)
tangency_long_only = tangency_portfolio(expected_returns, covariance, risk_free, long_only=True)
tangency_return = tangency @ expected_returns
tangency_volatility = np.sqrt(portfolio_variance(tangency, covariance))
tangency_sharpe = portfolio_sharpe(tangency, expected_returns, covariance, risk_free)

sigma_grid = np.linspace(0, 0.40, 100)
cml = risk_free + tangency_sharpe * sigma_grid
print("Carteira tangente:", tangency)
print("Sharpe tangente:", tangency_sharpe)
print("Carteira tangente long-only:", tangency_long_only)

plt.figure(figsize=(8, 4))
plt.plot(sigma_grid, cml, label="CML")
plt.scatter([tangency_volatility], [tangency_return], label="Tangência")
plt.xlabel("Volatilidade")
plt.ylabel("Retorno esperado")
plt.legend()
plt.title("Capital Market Line: retorno versus volatilidade")
plt.show()

## 6.6 — CAPM, beta, Security Market Line e alpha

O CAPM afirma

$$E(R_i)=R_f+\beta_i[E(R_m)-R_f].$$

A Security Market Line relaciona **retorno esperado e beta**:

$$SML(\beta)=R_f+\beta(E(R_m)-R_f).$$

A SML não usa volatilidade no eixo horizontal. Beta mede risco sistemático, não volatilidade total; alpha mede retorno não explicado pelo CAPM.

**Exercício:** compare dois ativos com o mesmo beta e volatilidades residuais diferentes. Eles têm o mesmo retorno requerido pelo CAPM, mas riscos totais diferentes.

In [ ]:
market_expected_return = 0.09
beta_values = np.linspace(-0.5, 2.0, 200)
sml = risk_free + beta_values * (market_expected_return - risk_free)
required_return = capm_expected_return(risk_free, 1.5, market_expected_return)

plt.figure(figsize=(8, 4))
plt.plot(beta_values, sml)
plt.scatter([1.5], [required_return])
plt.xlabel("Beta")
plt.ylabel("Retorno esperado")
plt.title("Security Market Line: retorno versus beta")
plt.show()
print("Retorno requerido para beta 1.5:", required_return)

## 6.7 — Teste empírico do CAPM e extensões multifatoriais

Estimamos alpha e beta com dados simulados. Um teste de alpha pergunta se o intercepto difere de zero; um beta significativo indica exposição ao mercado. Um $R^2$ baixo pode ser compatível com beta relevante e alto risco residual.

Extensões multifatoriais substituem o único fator de mercado por vários fatores de risco. Isso pode reduzir o risco residual, mas aumenta parâmetros e exige cuidado com data mining.

**Exercício:** adicione um fator irrelevante e compare alpha, $R^2$ ajustado e erro residual.

In [ ]:
rng = np.random.default_rng(123)
observations = 1_000
market_returns = rng.normal(0.0004, 0.012, observations)
asset_returns = 0.0001 + 1.2 * market_returns + rng.normal(0, 0.008, observations)

capm_estimate = estimate_capm(asset_returns, market_returns)
capm_model = sm.OLS(asset_returns, sm.add_constant(market_returns)).fit()
print("Alpha:", capm_estimate["alpha"])
print("Beta:", capm_estimate["beta"])
print("R²:", capm_estimate["r_squared"])
print("Risco residual:", capm_estimate["residual_risk"])
print("P-valor alpha statsmodels:", capm_model.pvalues[0])
print("Jensen alpha:", jensen_alpha(asset_returns, market_returns))

factor_data = pd.DataFrame({
    "mercado": market_returns,
    "valor": rng.normal(0, 0.01, observations),
    "momentum": rng.normal(0, 0.008, observations),
})
multifactor_asset = (
    0.0001
    + 1.1 * factor_data["mercado"]
    + 0.4 * factor_data["valor"]
    - 0.2 * factor_data["momentum"]
    + rng.normal(0, 0.006, observations)
)
multifactor_model = sm.OLS(multifactor_asset, sm.add_constant(factor_data)).fit()
print("\nExtensão multifatorial:")
print(multifactor_model.params)
print("R² ajustado multifatorial:", multifactor_model.rsquared_adj)

## 6.8 — Performance ajustada ao risco

As métricas respondem a perguntas diferentes:

- **Sharpe:** excesso de retorno por volatilidade total;
- **Sortino:** excesso de retorno por downside deviation;
- **Omega:** ganhos esperados divididos por perdas esperadas acima de um threshold;
- **Kappa:** excesso de retorno dividido por um lower partial moment de ordem $n$;
- **Information Ratio:** retorno ativo por tracking error;
- **Jensen alpha:** retorno além do previsto pelo CAPM;
- **Treynor:** excesso de retorno por beta sistemático, conforme esta definição do material;
- **certain equivalent:** média penalizada pela variância sob média-variância.

Duas carteiras podem ter Sharpe semelhante, mas Sortino e Omega diferentes quando sua assimetria e downside risk diferem.

**Exercício:** construa duas carteiras com a mesma média e desvio padrão, mas caudas opostas, e compare as métricas.

In [ ]:
performance_threshold = 0.005
positive_skew = np.array([0.01 - 0.02 / np.sqrt(10)] * 9 + [0.01 + 9 * 0.02 / np.sqrt(10)])
negative_skew = np.array([0.01 - 9 * 0.02 / np.sqrt(10)] + [0.01 + 0.02 / np.sqrt(10)] * 9)
benchmark_returns = np.full(positive_skew.size, 0.005)

performance_table = pd.DataFrame({
    "assimetria positiva": [
        sharpe_ratio(positive_skew, periods=1),
        sortino_ratio(positive_skew, threshold=performance_threshold, periods=1),
        omega_ratio(positive_skew, threshold=performance_threshold),
        kappa_ratio(positive_skew, threshold=performance_threshold, periods=1),
        information_ratio(positive_skew, benchmark_returns, periods=1),
        certainty_equivalent(positive_skew, risk_aversion=3.0),
    ],
    "assimetria negativa": [
        sharpe_ratio(negative_skew, periods=1),
        sortino_ratio(negative_skew, threshold=performance_threshold, periods=1),
        omega_ratio(negative_skew, threshold=performance_threshold),
        kappa_ratio(negative_skew, threshold=performance_threshold, periods=1),
        information_ratio(negative_skew, benchmark_returns, periods=1),
        certainty_equivalent(negative_skew, risk_aversion=3.0),
    ],
}, index=["Sharpe", "Sortino", "Omega", "Kappa", "Information Ratio", "Certain equivalent"])
display(performance_table)
print("Treynor da assimetria positiva com beta 1.2:", treynor_ratio(positive_skew, beta=1.2, periods=1))

### Autocorrelação e anualização ingênua do Sharpe

A regra $S_{anual}=S_{diário}\sqrt{252}$ pressupõe, entre outras coisas, ausência de autocorrelação. Com autocorrelação positiva, a variância de uma soma inclui covariâncias temporais:

$$\operatorname{Var}\left(\sum_tR_t\right)=\sum_t\operatorname{Var}(R_t)+2\sum_{i<j}\operatorname{Cov}(R_i,R_j).$$

Ignorar esses termos tende a subestimar o risco agregado e superestimar o Sharpe anualizado.

In [ ]:
rho = 0.3
eps = rng.normal(0, 0.01, 3_000)
autocorrelated_returns = np.zeros_like(eps)
for index in range(1, len(autocorrelated_returns)):
    autocorrelated_returns[index] = rho * autocorrelated_returns[index - 1] + eps[index]

daily_sharpe = autocorrelated_returns.mean() / autocorrelated_returns.std(ddof=1)
naive_annualized = daily_sharpe * np.sqrt(252)
blocks = autocorrelated_returns[:(len(autocorrelated_returns) // 252) * 252].reshape(-1, 252).sum(axis=1)
block_annualized = blocks.mean() / blocks.std(ddof=1)
print("Sharpe anualizado ingênuo:", naive_annualized)
print("Sharpe calculado em blocos anuais:", block_annualized)
print("Autocorrelação lag 1:", np.corrcoef(autocorrelated_returns[:-1], autocorrelated_returns[1:])[0, 1])

## Exercícios integradores

1. Compare utilidade exponencial, ARA, RRA e certainty equivalent para diferentes aversões.
2. Valide numericamente a fórmula do GMV e compare com a solução long-only.
3. Construa a fronteira eficiente analítica e com `scipy.optimize`.
4. Compare short selling, limites individuais e retorno-alvo.
5. Compare carteira GMV, carteira tangente, CML e SML.
6. Estime beta, alpha, $R^2$ e risco residual para um ativo simulado.
7. Compare beta com volatilidade e correlação.
8. Compare Sharpe, Sortino, Omega e Kappa em carteiras com assimetrias opostas.
9. Calcule Jensen alpha, Information Ratio e Treynor.
10. Mostre como autocorrelação positiva afeta a anualização ingênua do Sharpe.
11. Adicione fatores e avalie se o risco residual diminui.

## Leitura crítica dos resultados

Uma carteira ótima é ótima apenas sob as estimativas e restrições escolhidas. Compare pesos, retorno, volatilidade, concentração e sensibilidade a parâmetros. CML e SML respondem a perguntas diferentes; Sharpe, Sortino e Omega também. Termine sempre perguntando qual hipótese, se alterada, mudaria a decisão.